# Lesson 4.8: Polars and DuckDB

**Analyst:** Meron Welderufael
**Date:** 11/10/2026
**Notebook purpose:** Same per-tier FinFlow analysis in three tools.
Pandas (familiar), Polars (the faster DataFrame), DuckDB (SQL inside
Python). Benchmark each. Build the foundation for using the right tool
on the next engagement.

In [1]:
import polars as pl
import duckdb
import pyarrow
print(f"Polars: {pl.__version__}")
print(f"DuckDB: {duckdb.__version__}")
print(f"PyArrow: {pyarrow.__version__}")

Polars: 2.1.0
DuckDB: 2.0.0
PyArrow: 26.0.1


In [2]:
import pandas as pd
import polars as pl
import duckdb
import time

# Load with Pandas (the familiar starting point)
df_pandas = pd.read_csv('finflow-enriched-200k.csv',
                       parse_dates = ['signup_date', 'last_login_date', 'churn_date'])
print(f'shape: {df_pandas.shape}')
print(f"Memory: {df_pandas.memory_usage(deep=True).sum() / 1e6:.1f} MB")

shape: (200000, 22)
Memory: 57.7 MB


In [8]:
start = time.time()

pandas_result = df_pandas.groupby('marketing_name').agg(
    customer_count = ('customer_id', 'count'),
    total_mrr = ('monthly_fee', 'sum'),
    avg_sessions = ('total_sessions', 'mean')   
).round(2)

pandas_time = time.time() - start
print(f"Pandas time: {pandas_time:.4f}s")
print(pandas_result)

Pandas time: 0.1901s
                 customer_count   total_mrr  avg_sessions
marketing_name                                           
Enterprise Plus           20106  4001094.00        201.22
Professional              59878  1795741.22        182.60
Starter Plan             120016  1198959.84        163.94


In [4]:
df_polars = pl.read_csv('finflow-enriched-200k.csv',
                       try_parse_dates=True)

print(f"Shape: {df_polars.shape}")
print(f"Memory: {df_polars.estimated_size() / 1e6:.1f} MB")

Shape: (200000, 22)
Memory: 33.2 MB


In [9]:
start = time.time()

polars_result = (df_polars.group_by('marketing_name').agg([
    pl.col('customer_id').count().alias('customer_count'),
    pl.col('monthly_fee').sum().alias('total_mrr'),
    pl.col('total_sessions').mean().alias('avg_sessions')   
])
    .sort('marketing_name'))
polars_time = time.time() - start
print(f"Polars time: {polars_time:.4f}s")
print(polars_result)

Polars time: 0.0186s
shape: (3, 4)
┌─────────────────┬────────────────┬────────────┬──────────────┐
│ marketing_name  ┆ customer_count ┆ total_mrr  ┆ avg_sessions │
│ ---             ┆ ---            ┆ ---        ┆ ---          │
│ str             ┆ u32            ┆ f64        ┆ f64          │
╞═════════════════╪════════════════╪════════════╪══════════════╡
│ Enterprise Plus ┆ 20106          ┆ 4.001094e6 ┆ 201.223018   │
│ Professional    ┆ 59878          ┆ 1.7957e6   ┆ 182.603694   │
│ Starter Plan    ┆ 120016         ┆ 1.1990e6   ┆ 163.935867   │
└─────────────────┴────────────────┴────────────┴──────────────┘


In [14]:
start = time.time()

duckdb_result = duckdb.sql("""
   select
      marketing_name,
      count(*) AS customer_count,
      sum(monthly_fee)as total_mrr,
      avg(total_sessions) as avg_Sessions
   from df_pandas
   group by marketing_name
   order by marketing_name
""").df()

duckdb_time = time.time() - start
print(f"DuckDB time: {duckdb_time:.4f}s")
print(duckdb_result)

DuckDB time: 0.0247s
    marketing_name  customer_count     total_mrr  avg_Sessions
0  Enterprise Plus           20106  4.001094e+06    201.223018
1     Professional           59878  1.795741e+06    182.603694
2     Starter Plan          120016  1.198960e+06    163.935867


In [16]:
print("Per-tier customer count, three ways:\n")
print(f"Pandas  ({pandas_time:.4f}s):")
print(pandas_result[['customer_count']])
print(f"\nPolars  ({polars_time:.4f}s):")
print(polars_result.select(['marketing_name', 'customer_count']))
print(f"\nDuckDB  ({duckdb_time:.4f}s):")
print(duckdb_result[['marketing_name', 'customer_count']])

Per-tier customer count, three ways:

Pandas  (0.1901s):
                 customer_count
marketing_name                 
Enterprise Plus           20106
Professional              59878
Starter Plan             120016

Polars  (0.0186s):
shape: (3, 2)
┌─────────────────┬────────────────┐
│ marketing_name  ┆ customer_count │
│ ---             ┆ ---            │
│ str             ┆ u32            │
╞═════════════════╪════════════════╡
│ Enterprise Plus ┆ 20106          │
│ Professional    ┆ 59878          │
│ Starter Plan    ┆ 120016         │
└─────────────────┴────────────────┘

DuckDB  (0.0247s):
    marketing_name  customer_count
0  Enterprise Plus           20106
1     Professional           59878
2     Starter Plan          120016


**Observation:** Three tools, same answer. Pandas is the most readable
for someone who already knows it. Polars is fastest and reads cleanly
once you've seen the expression syntax a few times. DuckDB is the
right reach when the question is naturally SQL-shaped (joins, window
functions, complex aggregations).

In [17]:
# Save the Pandas DataFrame as Parquet
df_pandas.to_parquet('finflow-customers.parquet')

# File sizes for comparison
import os
csv_size = os.path.getsize('finflow-enriched-200k.csv') / 1e6
pq_size = os.path.getsize('finflow-customers.parquet') / 1e6
print(f"CSV size: {csv_size:.1f} MB")
print(f"Parquet size: {pq_size:.1f} MB")
print(f"Parquet is {csv_size / pq_size:.1f}x smaller")

CSV size: 36.9 MB
Parquet size: 7.1 MB
Parquet is 5.2x smaller


In [18]:
start = time.time()

parquet_result = duckdb.sql("""
    SELECT
        marketing_name,
        COUNT(*) AS customer_count,
        SUM(monthly_fee) AS total_mrr,
        AVG(total_sessions) AS avg_sessions
    FROM 'finflow-customers.parquet'
    where active = true
    GROUP BY marketing_name
    ORDER BY marketing_name
""").df()

parquet_time = time.time() - start
print(f"DuckDB-on-Parquet time: {parquet_time:.4f}s")
print(parquet_result)

DuckDB-on-Parquet time: 0.0934s
    marketing_name  customer_count     total_mrr  avg_sessions
0  Enterprise Plus           19108  3.802492e+06    210.399362
1     Professional           50982  1.528950e+06    209.995312
2     Starter Plan           89912  8.982209e+05    210.315119


In [ ]:
# Awareness-only code. Don't run this cell without a real PostgreSQL
# connection. The connection string and credentials are illustrative.

from sqlalchemy import create_engine

engine = create_engine('')

df = pd.read_sql("""
     SELECT * 
     FROM customers 
     WHERE signup_date >= '2024-01-01'
""", engine)

In [19]:
median_result = duckdb.sql("""
    SELECT marketing_name, 
    MEDIAN(monthly_fee) AS median_fee
    FROM df_pandas
    GROUP BY marketing_name
""").df()

print(median_result)

    marketing_name  median_fee
0  Enterprise Plus      199.00
1     Professional       29.99
2     Starter Plan        9.99
